In [ ]:

!pip -q install -U langchain langchain-community langchain-openai langchain-chroma langchain-text-splitters chromadb pypdf gradio


In [ ]:

from langchain_community.document_loaders import PyPDFLoader

pdf_path = "/content/the_nestle_hr_policy_pdf_2012.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Total PDF pages:", len(documents))
print(documents[0].page_content[:500])


In [ ]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0].page_content[:500])


In [ ]:
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
print("API key loaded successfully!" if api_key else "API key missing")

In [ ]:

!pip -q install sentence-transformers


In [ ]:

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="nestle_hr_policy_free"
)

print("ChromaDB created successfully!")
print("Total chunks stored:", vector_db._collection.count())


In [ ]:

def retrieve_hr_policy(question):
    results = vector_db.similarity_search(question, k=3)

    context = "\n\n".join(
        doc.page_content for doc in results
    )

    return context

question = "What is Nestle's HR policy about?"

context = retrieve_hr_policy(question)

print("Relevant HR policy information:\n")
print(context[:1500])


In [ ]:

%pip install -q --upgrade --force-reinstall "huggingface-hub==0.36.2" "transformers==4.57.6" "gradio==5.49.1" sentencepiece


In [ ]:

import gradio as gr
from transformers import pipeline
import huggingface_hub

print("Gradio:", gr.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("Imports successful!")


In [ ]:

import gradio as gr
from transformers import pipeline

# Load the free AI model
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=-1
)

def hr_chatbot(message, history):
    if not message.strip():
        return "Please enter an HR policy question."

    # Find relevant information in the PDF
    docs = vector_db.similarity_search(message, k=2)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    # Create an AI prompt
    prompt = f"""Answer the question using only the context below.
If the answer is not available, say:
I could not find that information in the HR policy.

Context:
{context[:1500]}

Question: {message}
Answer:"""

    # Generate answer
    response = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        truncation=True
    )

    answer = response[0]["generated_text"].strip()

    if not answer:
        answer = "I could not find that information in the HR policy."

    pages = sorted(set(
        doc.metadata.get("page", 0) + 1
        for doc in docs
    ))

    return answer + "\n\nSource pages: " + ", ".join(map(str, pages))

# Build the chatbot interface
demo = gr.ChatInterface(
    fn=hr_chatbot,
    title="Nestlé AI-Powered HR Assistant",
    description="Ask questions about Nestlé's HR Policy.",
    examples=[
        "What is Nestlé's HR policy about?",
        "What are Nestlé's HR principles?",
        "What does the policy say about employees?"
    ]
)

demo.launch(share=True)


In [ ]:

!wget -q https://raw.githubusercontent.com/2310030022-coder/Nestle-AI-HR-Assistant/main/app.py -O app.py
print("Application downloaded successfully!")


In [ ]:

!python app.py


In [ ]:

!wget -q https://raw.githubusercontent.com/2310030022-coder/Nestle-AI-HR-Assistant/main/app.py -O app.py
!python app.py


In [ ]:

from pathlib import Path

app = Path("/content/app.py")
code = app.read_text()

start = code.index("    examples=[", code.index("demo = gr.ChatInterface("))
end = code.index("\n    ]", start) + len("\n    ]")

code = code[:start] + """    examples=[
        ["What is Nestlé's HR policy about?", None],
        ["What are the HR principles?", None],
        ["What does the policy say about employees?", None]
    ]""" + code[end:]

app.write_text(code)
print("Gradio examples fixed successfully!")


In [ ]:

!python app.py
